# One-macro adapter training with wider square bands

Use a Colab GPU. This retry uses eight square bands with half-widths **6, 12, 18, 24, 30, 36, 42, and 48**. It keeps the same six training goals, one-macro horizon and execution stride, success-only time-order labels, and cumulative fitting procedure.

Band 1 refits from identity using the same **36 archived raw-MSE rollouts** and their saved goal tensors. The later bands collect **204 new rollouts** by default, using the preceding checkpoint. Each strip draws up to six starts per goal and skips goal–band combinations with no legal reset area. The outer square spans the right room.

Run through the Band 1 check, then continue with the wider bands. This run has a separate Drive output folder. Set `RESUME_RUN_ID` to resume.

In [ ]:
from google.colab import drive
from pathlib import Path
from datetime import datetime,timezone
import hashlib,json,shutil,signal,subprocess,sys,time
import torch

drive.mount('/content/drive')
assert torch.cuda.is_available(), 'Select a GPU runtime, then Run all.'
DRIVE=Path('/content/drive/MyDrive')
SOURCE_RUN=DRIVE/'datasets/curriculum_cost_adapter/radial_multigoal_ranking_v1/20261106T195304628603Z'
assert (SOURCE_RUN/'branches/multi/complete.json').is_file(), 'Set SOURCE_RUN to your completed multi-goal ranking run.'
RADIAL_RUN=DRIVE/'datasets/curriculum_cost_adapter/experience_temporal_v1/20261107T142845862912Z'
RAW_BAND1_RUN=DRIVE/'datasets/curriculum_cost_adapter/raw_mse_half_square_h1_v1/20261108T083911741327Z'
RESUME_RUN_ID=None  # Set to the printed ID to resume an interrupted run.
STARTS_PER_GOAL=6
RUN_ID=RESUME_RUN_ID or datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUTPUT=DRIVE/'datasets/curriculum_cost_adapter/experience_one_macro_adapter_wide_v1'/RUN_ID
REPO=Path('/content/i-jepa-wm-experience-one-macro-adapter-wide')
REPO_URL='https://github.com/kmylonas/i-jepa-wm.git'
SOURCE_COMMIT='9efadffa2c4364fe8e656a7b421eaa9fb6f87a06'
DINO_ROOT=Path('/content/dino_wm')
DINO_COMMIT='0a9492fa12044b852ae9e001cc74604b79c8bb0c'
ENCODER_DRIVE=DRIVE/'checkpoints/IN1K-vit.h.14-300e.encoder-only.pth.tar'
WM_DRIVE=DRIVE/'datasets/wall_matched_fixed_multi_v3/fixed/world_model/best.pt'
ARTIFACTS=Path('/content/experience-one-macro-adapter-wide-artifacts')
ENCODER=ARTIFACTS/'encoder.pth.tar'
CHECKPOINT=ARTIFACTS/'world_model.pt'
ENCODER_SHA='3f37d8a7572e25c2dce9a3072c2fd8cd55ce32dc6edb92a8996b0e4cb140bf98'
WM_SHA='b0f3ccfefe0534d9f51af2d0767619913f48d0c323ebb323f0bcd4b800206c3f'

def sha256(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda:stream.read(8*1024*1024),b''):h.update(block)
    return h.hexdigest()

def shell(command,cwd=None):
    command=list(map(str,command));print('$',' '.join(command),flush=True)
    with subprocess.Popen(command,cwd=cwd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1) as process:
        try:
            for line in process.stdout:print(line,end='',flush=True)
            code=process.wait()
        except BaseException:
            process.send_signal(signal.SIGINT)
            try:process.wait(timeout=10)
            except subprocess.TimeoutExpired:process.terminate()
            raise
    if code:raise subprocess.CalledProcessError(code,command)

print('GPU:',torch.cuda.get_device_name())
print('Frozen ranking source:',SOURCE_RUN)
print('Run ID:',RUN_ID)
print('New experience curriculum results:',OUTPUT)

## Load the pinned code and frozen models

The world-model checkpoint, encoder and renderer are the same ones used in the previous planning comparison.

If Drive interrupts a checksum read, the code retries and prints the file path. For a persistent error, remount Drive and rerun the affected band.

In [ ]:
if not REPO.exists():shell(['git','clone','--branch','curriculum-cost-adapter','--single-branch',REPO_URL,REPO])
else:
    assert subprocess.check_output(['git','-C',str(REPO),'remote','get-url','origin'],text=True).strip()==REPO_URL
    assert not subprocess.check_output(['git','-C',str(REPO),'status','--porcelain'],text=True).strip(), 'Clone has local changes.'
    shell(['git','-C',REPO,'fetch','origin','curriculum-cost-adapter'])
shell(['git','-C',REPO,'checkout','--detach',SOURCE_COMMIT])
shell([sys.executable,'-m','pip','install','-q','gym==0.26.2','omegaconf','pyyaml','scipy','einops','matplotlib','pillow','imageio','imageio-ffmpeg'])
if not DINO_ROOT.exists():shell(['git','clone','https://github.com/gaoyuezhou/dino_wm.git',DINO_ROOT])
assert not subprocess.check_output(['git','-C',str(DINO_ROOT),'status','--porcelain','--untracked-files=no'],text=True).strip(), 'Renderer has tracked changes.'
shell(['git','-C',DINO_ROOT,'checkout',DINO_COMMIT])
ARTIFACTS.mkdir(parents=True,exist_ok=True)
for source,target,expected in ((ENCODER_DRIVE,ENCODER,ENCODER_SHA),(WM_DRIVE,CHECKPOINT,WM_SHA)):
    assert source.is_file(), f'Missing frozen artifact: {source}'
    if not target.exists() or sha256(target)!=expected:shutil.copy2(source,target)
    assert sha256(target)==expected, f'Unexpected checkpoint: {source}'

def command(operation, band=None):
    args=[sys.executable,'-m','src.mylonas_ijepa_wm.experience_curriculum',operation,
          '--source-run',SOURCE_RUN,'--output-dir',OUTPUT,'--world-model-checkpoint',CHECKPOINT,
          '--encoder-checkpoint',ENCODER,'--dino-wm-root',DINO_ROOT,'--device','cuda',
          '--starts-per-goal',STARTS_PER_GOAL,'--labels','temporal','--sampling','half-square','--planning-horizon',1,
          '--actions-per-plan',1,'--wide-square-bands','--band1-source',RAW_BAND1_RUN]
    if band is not None:
        args+=['--band',band]
    return args

def run_band(band):
    started=time.monotonic()
    shell(command('stage',band),cwd=REPO)
    print('Band minutes:',round((time.monotonic()-started)/60,2))

## Fit Band 1 from the saved raw rollouts

In [ ]:
run_band(1)

## Check what the first checkpoint learned

Near-goal accuracy compares held-out positions within 8.5 units of each withheld goal, the approximate distance to a corner of a 6-unit half-width square. A ranking pair needs a physical distance gap of at least 0.5 units. The physical distances are used here for evaluation. Compare this local score with the full-room score, where most positions are farther than the experience used for this checkpoint.

In [ ]:
shell(command('compare',1),cwd=REPO)
first=json.loads((OUTPUT/'band_1_comparison.json').read_text())
training=json.loads((OUTPUT/'band_1/training.json').read_text())

import numpy as np
import pandas as pd

collected=pd.DataFrame([dict(goal=tuple(row['case']['goal']),rollouts=1,successes=int(row['success']),
    actions=row['steps'],retained=row['retained_pairs']) for row in first['checkpoints']['band_1']['collection']])
print('Training pairs from the archived rollouts')
display(collected.groupby('goal',sort=False).sum(numeric_only=True))
print('Cumulative retained pairs:',first['checkpoints']['band_1']['retained_pairs'])
print('Training probe before and after fitting')
display(pd.DataFrame([training['probe_before'],training['probe_after']],index=['before','after']).round(5))

def near_accuracy(row,heldout):
    distance=np.asarray(row['true_distances'])
    selected=np.asarray(heldout,dtype=bool)&(distance<=8.5)
    d=distance[selected]
    c=np.asarray(row['adapted_costs'])[selected]
    far,near=np.where(d[:,None]-d[None,:]>=.5)
    if not len(far):
        return 0,None
    tolerance=max(1e-7,.001*float(np.ptp(c)))
    return len(far),100*float(((c[far]-c[near])>tolerance).mean())

rows=[]
for method in ('raw','band_1'):
    for row in first['methods'][method][1:]:
        pairs,pct=near_accuracy(row,first['heldout_mask'])
        rows.append(dict(method=method,goal=tuple(row['goal']),near_pairs=pairs,near_correct_pct=pct,
            full_room_correct_pct=100*row['adapted_rank']['correct_fraction'],
            full_room_uphill_pct=100*row['heldout_directional']['adapted_up_fraction']))
scores=pd.DataFrame(rows)
print('Mean held-out scores across goals')
display(scores.groupby('method',sort=False).mean(numeric_only=True).round(2))
print('Near-goal scores and pair counts by withheld goal')
display(scores[['method','goal','near_pairs','near_correct_pct']].round(2))
scores.to_csv(OUTPUT/'band_1_heldout_scores.csv',index=False)

## Continue with wider square strips

The first new strip spans half-widths 6–12. Each subsequent strip adds six units. A band with zero eligible pairs carries forward its incoming adapter. The final cells report collection yield and full right-room ranking.

## Band 2 · square half-width 6–12

In [ ]:
run_band(2)

## Band 3 · square half-width 12–18

In [ ]:
run_band(3)

## Band 4 · square half-width 18–24

In [ ]:
run_band(4)

## Band 5 · square half-width 24–30

In [ ]:
run_band(5)

## Band 6 · square half-width 30–36

In [ ]:
run_band(6)

## Band 7 · square half-width 36–42

In [ ]:
run_band(7)

## Band 8 · square half-width 42–48

In [ ]:
run_band(8)

## Compare the learned geometry

The first table uses the entire held-out right-room grid. The second uses the smaller region shared with the radial run.

In [ ]:
shell(command('compare'),cwd=REPO)
report=json.loads((OUTPUT/'comparison.json').read_text())

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Circle

metrics=[]
for method, rows in report['methods'].items():
    for goal_id, row in enumerate(rows):
        order=row['adapted_rank']
        steps=row['heldout_directional']
        metrics.append(dict(method=method,goal_id=goal_id,goal=tuple(row['goal']),
                            goal_type='withheld' if goal_id else 'seen',queries=row['heldout_count'],
                            correct_order=order['correct_fraction'],pair_ties=order['tie_fraction'],
                            uphill=steps['adapted_up_fraction'],step_ties=steps['adapted_tie_fraction']))
table=pd.DataFrame(metrics)
columns=['correct_order','pair_ties','uphill','step_ties']
print('Full right room · average per withheld goal (percent)')
display((100*table[table.goal_type=='withheld'].groupby('method',sort=False)[columns].mean()).round(2))
print('Per-goal results (fractions)')
display(table.round(4))
table.to_csv(OUTPUT/'per_goal_metrics.csv',index=False)

collected=pd.DataFrame([
    dict(band=band,goal=tuple(row['case']['goal']),
         rollouts=1,successes=int(row['success']),steps=row['steps'],stationary=row['stationary_steps'],
         failed_skipped=int(row['skipped_failed']),eligible=row['eligible_pairs'],retained=row['retained_pairs'])
    for band,stage in report['checkpoints'].items() for row in stage['collection']
])
print('Collection totals per band')
display(collected.groupby('band').sum(numeric_only=True))
print('Collection totals per goal and band')
display(collected.groupby(['band','goal']).sum(numeric_only=True))
collected.to_csv(OUTPUT/'collection_counts.csv',index=False)
print('Optimizer updates by band')
display(pd.Series({band:s['updates'] for band,s in report['checkpoints'].items()},name='updates'))

def old_cohort(methods, label):
    return [dict(recipe=label,checkpoint=name,
                 correct_order=100*np.mean([r['adapted_rank']['correct_fraction'] for r in rows[1:]]),
                 uphill=100*np.mean([r['heldout_directional']['adapted_up_fraction'] for r in rows[1:]]))
            for name,rows in methods.items()]

legacy=old_cohort(report['legacy_methods'],'square')
radial_report=RADIAL_RUN/'comparison.json'
if radial_report.is_file():
    previous=json.loads(radial_report.read_text())
    new_config=json.loads((OUTPUT/'experiment.json').read_text())
    old_config=json.loads((RADIAL_RUN/'experiment.json').read_text())
    if (all(old_config[k]==new_config[k] for k in ('handcrafted','fitting','scale'))
            and old_config['manifest']['goal_ids']==new_config['manifest']['goal_ids']
            and all(previous[k]==report[k] for k in ('goal_ids','grid','heldout_mask'))):
        legacy=old_cohort(previous['methods'],'radial')+legacy
print('Original restricted cohort · mean per withheld goal (percent)')
display(pd.DataFrame(legacy).round(2))

## Full right-room heatmaps and collection coverage

Each panel uses its own unclipped cost scale. Dashed squares mark the band boundaries around the plotted goal. Orange lines highlight the inner and outer edges of the panel's band. Boundaries outside the plot are clipped. The squares mark start bands for collection goals and serve as distance references for held-out goals. Observations can come from anywhere along a successful path. Set `HEATMAP_GOALS` to view other goals.

The separate coverage plots show the actual starts and retained observations for each stage.

In [ ]:
manifest = json.loads((OUTPUT / 'experiment.json').read_text())['manifest']
PLOT_DIR = OUTPUT / 'heatmaps'

from IPython.display import Image, display
from matplotlib import patheffects
from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle
import matplotlib.pyplot as plt
import numpy as np

HEATMAP_GOALS = [0, 15]
grid = report['grid']
shape = (len(grid['ys']), len(grid['xs']))
methods = list(report['methods'])
bands = manifest['bands']
collection_goals = {tuple(case['goal']) for stage in manifest['stages'] for case in stage}
PLOT_DIR.mkdir(parents=True, exist_ok=True)

for goal_id in HEATMAP_GOALS:
    goal = tuple(report['methods']['raw'][goal_id]['goal'])
    gx, gy = goal
    cols = 5
    rows = (len(methods) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(17, 4.8 * rows),
                             squeeze=False, constrained_layout=True)
    for ax in axes.flat[len(methods):]:
        ax.set_visible(False)
    for ax, method in zip(axes.flat, methods):
        row = report['methods'][method][goal_id]
        costs = np.asarray(row['adapted_costs'])[grid['point_ids']].reshape(shape)
        mesh = ax.pcolormesh(grid['xs'], grid['ys'], costs, shading='nearest', cmap='viridis',
                             vmin=0, vmax=max(float(costs.max()), 1e-9))
        active = ()
        title = {'raw': 'Raw MSE', 'handcrafted': 'Hand-crafted'}.get(method, method)
        if method.startswith('band_'):
            number = int(method.split('_')[1])
            active = bands[number - 1]
            title = f'Band {number} · {active[0]}–{active[1]}'
            if report['checkpoints'][method]['updates'] == 0:
                title += '\nCarried forward (0 updates)'
        for _, radius in bands:
            highlight = radius in active
            width = 1.6 if highlight else 0.8
            outline = Rectangle((gx - radius, gy - radius), 2 * radius, 2 * radius,
                                fill=False, edgecolor='#ffaf36' if highlight else 'white',
                                linestyle='-' if highlight else '--', linewidth=width,
                                zorder=4 if highlight else 3,
                                path_effects=[patheffects.withStroke(linewidth=width + 0.9,
                                                                      foreground='#282838')])
            ax.add_patch(outline)
        ax.scatter(*goal, marker='*', s=110, c='black', edgecolor='white', zorder=5)
        ax.set(title=title, xlabel='Ball x', xlim=(35, 60), ylim=(4, 60), aspect='equal')
        fig.colorbar(mesh, ax=ax, shrink=.6, label='Latent MSE')
    for ax in axes[:, 0]:
        ax.set_ylabel('Ball y')
    label = 'Collection start bands' if goal in collection_goals else 'Held-out goal · reference bands'
    fig.suptitle(f'Goal {goal_id} {goal} · {label}')
    fig.legend(handles=[Line2D([], [], color='#4e4e5e', linestyle='--', label='Square band boundaries'),
                        Line2D([], [], color='#d57900', label='Inner and outer edges of panel band')],
               loc='upper center', bbox_to_anchor=(.5, -.005), ncol=2, frameon=False)
    path = PLOT_DIR / f'goal_{goal_id}_bands.png'
    fig.savefig(path, dpi=130, bbox_inches='tight')
    plt.close(fig)
    display(Image(filename=str(path)))

In [ ]:
count=len(report['checkpoints'])
cols=4
rows=(count+cols-1)//cols
fig,axes=plt.subplots(rows,cols,figsize=(14,4.8*rows),sharey=True,constrained_layout=True)
for ax in axes.flat[count:]:
    ax.set_visible(False)
for ax,(band,stage) in zip(axes.flat,report['checkpoints'].items()):
    for row in stage['collection']:
        if row['retained_positions']:
            ax.scatter(*np.asarray(row['retained_positions']).T,s=6,c='tab:blue',alpha=.35)
        ax.scatter(*row['case']['start'],marker='x',s=22,c='tab:orange')
        ax.scatter(*row['case']['goal'],marker='*',s=50,c='black')
    ax.set(title=f"{band.replace('_',' ').title()} · {stage['retained_pairs']} pairs",
           xlim=(35,60),ylim=(4,60),aspect='equal',xlabel='Ball x')
for ax in axes[:,0]:
    ax.set_ylabel('Ball y')
fig.suptitle('Blue dots are retained observations, orange crosses are starts, black stars are collection goals')
fig.savefig(OUTPUT/'coverage.png',dpi=130,bbox_inches='tight')
plt.show()
plt.close(fig)